# ER-MAP — Doctor Agent GRPO Training (Kaggle Free-Tier)

**Target hardware:** Tesla T4 16 GB (or P100 16 GB) — Kaggle's free GPU.

**What this notebook does:**
1. Clones / mounts the ER-MAP repo
2. Installs the missing pieces (Unsloth, TRL, Groq, HF Hub) on top of Kaggle's pre-baked PyTorch image
3. Loads Llama-3.1-8B in 4-bit + LoRA(r=16) via Unsloth (~7 GB VRAM)
4. Runs the manual GRPO loop from `ER_MAP/training/train_grpo.py` with 3-phase curriculum learning
5. Pushes LoRA adapter checkpoints to a Hugging Face Hub repo every 20 episodes so the 12-hour Kaggle session limit doesn't lose progress

**Required Kaggle Secrets** (Add-ons → Secrets):
- `GROQ_NURSE_API_KEY`, `GROQ_PATIENT_API_KEY`, `GROQ_EMPATHY_JUDGE_API_KEY`, `GROQ_MEDICAL_JUDGE_API_KEY` — for the multi-agent env actors and judges
- `HF_TOKEN` — to push checkpoints (use a fine-grained write token)
- `WANDB_API_KEY` — *optional*, for the reward-growth chart

**Notebook settings (right sidebar):**
- Accelerator: **GPU T4 x2** (or P100)
- Internet: **On** *(Groq calls require this)*
- Persistence: Files only

## 1 · Sanity check the GPU + clone the repo

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
!python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
# --- OPTION A: clone the public repo (preferred) ----------------------
# Replace <your-github-fork> with your actual fork URL. Public fork
# works without any token. For a private repo, set HF_TOKEN OR pass
# a GH PAT via Kaggle Secrets.
GIT_URL = "https://github.com/<your-github-fork>/Meta_Finals.git"
BRANCH  = "main"
REPO_ROOT = "/kaggle/working/Meta_Finals"

import os, subprocess
if not os.path.isdir(f"{REPO_ROOT}/ER_MAP") and "<your-github-fork>" not in GIT_URL:
    print(subprocess.check_output(
        ["git", "clone", "--depth", "1", "-b", BRANCH, GIT_URL, REPO_ROOT],
        stderr=subprocess.STDOUT,
    ).decode())

# --- OPTION B: dataset upload (if you don't want to push to GitHub) ---
# 1. Locally: zip the repo (excluding .git, checkpoints, __pycache__).
# 2. Kaggle: New Dataset -> upload the zip -> name it `ermap-source`.
# 3. This notebook: Add Data -> ermap-source.
# 4. Run the next cell to copy /kaggle/input/ermap-source/ into
#    /kaggle/working/Meta_Finals (writeable).
DATASET_DIR = "/kaggle/input/ermap-source"
if not os.path.isdir(f"{REPO_ROOT}/ER_MAP") and os.path.isdir(DATASET_DIR):
    import shutil
    shutil.copytree(DATASET_DIR, REPO_ROOT, dirs_exist_ok=True)
    print(f"Copied {DATASET_DIR} -> {REPO_ROOT}")

assert os.path.isdir(f"{REPO_ROOT}/ER_MAP"), (
    "Repo not found. Either set GIT_URL above (Option A) or upload the "
    "repo as a Kaggle Dataset named 'ermap-source' (Option B)."
)
print("Repo ready at", REPO_ROOT)

## 2 · Install the missing dependencies

Kaggle's GPU image already ships with PyTorch 2.x + CUDA 12 + transformers + accelerate + peft + bitsandbytes. We only add Unsloth (which pins matching xformers/triton), TRL, Gymnasium, Groq SDK, and the HF Hub client.

In [ ]:
# --upgrade is critical: Kaggle's pre-baked layer often ships an
# OLD `unsloth` paired with whatever fresh `unsloth_zoo` pip pulled
# this morning, and the import then fails with:
#   ImportError: cannot import name 'create_gradient_checkpointing_buffer'
# Forcing both packages to upgrade in one resolve pass keeps them in lockstep.
!pip install -q --upgrade -r {REPO_ROOT}/kaggle/requirements_kaggle.txt
# Sanity check the unsloth import — it's the most fragile dep on Kaggle.
# If you see the gradient_checkpointing ImportError below, run:
#   !pip install --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo
# in a NEW cell, then RESTART the kernel and re-run from cell 2.
!python -c "import unsloth, unsloth_zoo; print('unsloth', unsloth.__version__, '| unsloth_zoo', unsloth_zoo.__version__)"

## 3 · Wire Kaggle Secrets into env vars

ER-MAP reads `GROQ_NURSE_API_KEY` / `GROQ_PATIENT_API_KEY` / etc. directly from `os.environ`. The helper below copies your Kaggle Secrets into those env vars in one shot.

In [ ]:
import sys, os
sys.path.insert(0, REPO_ROOT)               # so we can import ER_MAP
sys.path.insert(0, f"{REPO_ROOT}/kaggle")   # so we can import kaggle_helpers

from kaggle_helpers import (
    load_kaggle_secrets,
    kaggle_env_summary,
    push_checkpoint_to_hub,
    download_checkpoint_from_hub,
)

load_kaggle_secrets()
kaggle_env_summary()

## 4 · (Optional) Resume from a previous Kaggle session

If you've trained before and pushed an adapter to HF Hub, set `HF_RESUME_REPO` and run the cell to pull the latest LoRA adapter into `/kaggle/working/checkpoints/resume/`. The training cell will pick it up automatically.

In [ ]:
HF_PUSH_REPO   = "<your-username>/ermap-doctor-lora"   # where checkpoints will be pushed
HF_RESUME_REPO = ""  # e.g. "<your-username>/ermap-doctor-lora"; leave empty to start fresh

RESUME_DIR = "/kaggle/working/checkpoints/resume"
if HF_RESUME_REPO:
    download_checkpoint_from_hub(HF_RESUME_REPO, RESUME_DIR)
    print("Resume dir contents:", os.listdir(RESUME_DIR) if os.path.isdir(RESUME_DIR) else "(empty)")

## 5 · Configure the GRPO run

The defaults below are tuned for **one 12-hour Kaggle session** on a single T4. They produce a clean upward reward-growth curve through Phase 1 + early Phase 2; if you have a second session, lower `--episodes` is fine because LoRA adapters resume cleanly via Step 4 above.

| Parameter | Value | Reason |
|---|---|---|
| Model | `unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit` | Llama-3-family small-tier, 4-bit ~5 GB |
| LoRA rank | 16 | balances expressivity vs speed on T4 |
| Group size G | 2 | Kaggle's T4 fits G=2 comfortably; G=4 needs 30+ min/group |
| Episodes (cap) | 120 | hard cap; early-stop usually finishes first |
| LR | 5e-6 | conservative, prevents catastrophic forgetting on small group |
| KL beta | 0.04 | matches the paper's recipe; restrains drift from base policy |
| Max episode steps | 20 | matches `triage_env.py` default |
| Internal exchanges | 5 | shorter than default (8) to fit within 12 h budget |

### Train-until-optimal (per-phase reward thresholds)

Training **never** runs for a fixed episode budget. After every GRPO update we look at the last `CONVERGENCE_WINDOW=3` groups; if **all three** belong to the same current phase AND each has `rolling_avg_reward >= PHASE_REWARD_TARGETS[current_phase]`, we either:

- **Phase 1 / Phase 2** → force-promote to the next curriculum phase (the buffer is then cleared so stale entries don't satisfy the next phase's check).
- **Phase 3** → terminate training (this is the 'optimal rewards constantly received' criterion).

Why per-phase, not a single global bar? The phases are not equally difficult — Phase 1 wins are worth ~`+2.0` on the reward scale (full terminal_win on a clean SOAP) while Phase 3 wins routinely cost `~0.5` in consent / empathy friction even when the diagnosis is correct. A single global `+1.5` would either gate Phase 3 too aggressively or pass Phase 1 with garbage Phase-2 behavior.

| Phase | Default target | Why this number | Action when met |
|---|---|---|---|
| 1 — Tool Mastery | `+1.5` | A Phase-1 episode that uses tools cleanly + discharges with the correct treatment lands at `+1.6 .. +2.0`. Sustaining `+1.5` means the model has tool-format down. | force-promote to Phase 2 |
| 2 — Clinical Reasoning | `+1.2` | Phase 2 adds noisy SOAP and mixed compliance. A solid clinician policy lands at `+1.2 .. +1.5`. | force-promote to Phase 3 |
| 3 — Empathetic Negotiation | `+1.0` | Phase 3 imposes empathy + consent costs (`-0.3..-0.6` per episode even on wins). Sustained `+1.0` here is genuinely hard and is the hackathon success criterion. | END TRAINING |

| Knob | Default | Meaning |
|---|---|---|
| `PHASE_REWARD_TARGETS` | `{1: 1.5, 2: 1.2, 3: 1.0}` | per-phase sustained rolling-avg-reward bar |
| `PHASE_MIN_WIN_RATE` | `0.20` | soft floor on rolling win rate (sanity check) |
| `CONVERGENCE_WINDOW` | `3` | how many consecutive groups must hit the bar |
| `EARLY_STOP_ENABLED` | `True` | set `False` to always burn the full `NUM_EPISODES` budget |

### Estimated wall-clock per phase on Kaggle T4 ×2

Each episode = 6–14 env steps × (Doctor.generate ≈ 2–3 s) + 4–8 Groq calls (≈ 0.4–1.0 s each). One GRPO update over `G=2` trajectories = 1 forward + 1 backward over response tokens ≈ 60–120 s on T4. Net per-group wall-clock ≈ **8–12 minutes**.

| Phase | Typical episodes to reach target | Wall-clock | Notes |
|---|---|---|---|
| 1 (target `+1.5` × 3) | 16 – 30 episodes (8 – 15 groups) | **~1.5 – 2.5 h** | Easy patients + clean SOAP — tool-format is the only thing the model has to learn. |
| 2 (target `+1.2` × 3) | 24 – 40 episodes (12 – 20 groups) | **~2.0 – 3.5 h** | Mixed-compliance patients + noisy SOAP. Bulk of the policy improvement happens here. |
| 3 (target `+1.0` × 3) | 30 – 60 episodes (15 – 30 groups) | **~2.5 – 5.0 h** | Hard patients + empathy/consent costs. May not converge in 12 h on a fresh base; that's why `NUM_EPISODES=120` is the hard cap. |
| **Total** | 70 – 130 episodes | **~6 – 11 h** | Fits inside Kaggle's 12 h GPU session with ~1 h margin. |

If Phase 3 doesn't converge before the 12 h limit, your latest LoRA checkpoint is already on HF Hub (we push every 20 episodes), so just resume in a fresh session via `HF_RESUME_REPO`.

In [ ]:
# --- Training hyperparameters ---
MODEL_NAME       = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
NUM_EPISODES     = 120          # HARD CAP; early-stop usually finishes first
GROUP_SIZE       = 2
LEARNING_RATE    = 5e-6
KL_BETA          = 0.04
OUTPUT_DIR       = "/kaggle/working/er_map_grpo_checkpoints"
PUSH_EVERY_EPS   = 20
USE_WANDB        = bool(os.environ.get("WANDB_API_KEY"))

# --- Early-stopping (per-phase reward thresholds) ---
# After every GRPO update, we check the last CONVERGENCE_WINDOW groups.
# If ALL of them are in the SAME current phase AND each has
# rolling_avg_reward >= PHASE_REWARD_TARGETS[current_phase], we either:
#   - force-promote to the next phase (Phase 1, Phase 2), OR
#   - terminate training (Phase 3).
# Baseline (un-trained Groq Doctor) avg reward by phase:
#   P1=+0.76, P2=+0.59, P3=+0.39
# So the +1.5/+1.2/+1.0 bar = 2.0x / 2.0x / 2.6x improvement.
EARLY_STOP_ENABLED       = True
PHASE_REWARD_TARGETS     = {1: 1.5, 2: 1.2, 3: 1.0}
PHASE_MIN_WIN_RATE       = 0.20  # soft floor; +1.0 reward implies >=20% wins
CONVERGENCE_WINDOW       = 3     # 3 consecutive groups must qualify

# --- Per-episode budget controls (passed via env vars) ---
os.environ["ERMAP_MAX_EPISODE_STEPS"]      = "20"
os.environ["ERMAP_MAX_INTERNAL_EXCHANGES"] = "5"
# Doctor-on-Kaggle is the LOCAL trained model, NOT a Groq call. The
# Doctor's Groq key is therefore unused here, but Nurse / Patient /
# Empathy Judge / Medical Judge all hit Groq once per env step.
# Traffic-shaping: high-volume roleplay agents (Nurse + Patient) on the
# 8B-instant pool (500K TPD, 14,400 RPD); the two judges stay on 70B-
# versatile because their grading quality directly shapes the reward.
os.environ["ERMAP_NURSE_MODEL"]            = "llama-3.1-8b-instant"
os.environ["ERMAP_PATIENT_MODEL"]          = "llama-3.1-8b-instant"
os.environ["ERMAP_EMPATHY_JUDGE_MODEL"]    = "llama-3.3-70b-versatile"
os.environ["ERMAP_MEDICAL_JUDGE_MODEL"]    = "llama-3.3-70b-versatile"

# Sanity: at least one Groq key must be present, otherwise the env
# falls back to mock responses and the trained model won't see
# realistic dialogue.
assert any(
    os.environ.get(k) for k in [
        "GROQ_NURSE_API_KEY", "GROQ_PATIENT_API_KEY",
        "GROQ_EMPATHY_JUDGE_API_KEY", "GROQ_MEDICAL_JUDGE_API_KEY",
        "GROQ_API_KEY",
    ]
), ("No Groq key found in Kaggle Secrets — add at least "
    "GROQ_NURSE_API_KEY before running training.")
print("Hyperparameters and env vars set.")

## 6 · Dry-run smoke test (no GPU, no model load)

Verifies the curriculum scheduler + reward verifier + metrics logger are wired correctly **before** burning GPU minutes on a real run.

In [ ]:
from ER_MAP.training.train_grpo import train

_ = train(
    num_episodes=8,
    group_size=2,
    model_name=MODEL_NAME,
    learning_rate=LEARNING_RATE,
    kl_beta=KL_BETA,
    output_dir="/kaggle/working/_dryrun",
    dry_run=True,
)
print("Dry-run OK — scheduler + verifier wiring is healthy.")

## 7 · Wire periodic HF-Hub push into the training loop

We monkey-patch `save_lora_adapters` so every checkpoint dump also pushes to HF Hub. Failures are non-fatal — training keeps running even if the push fails.

In [ ]:
from ER_MAP.training import train_grpo as _tg
_original_save = _tg.save_lora_adapters
_episode_marker = {"n": 0}

def save_lora_adapters_with_push(model, tokenizer, output_dir):
    _original_save(model, tokenizer, output_dir)
    _episode_marker["n"] += 1
    if HF_PUSH_REPO and "<your-username>" not in HF_PUSH_REPO:
        try:
            push_checkpoint_to_hub(
                output_dir,
                HF_PUSH_REPO,
                commit_message=f"checkpoint @ {os.path.basename(output_dir)}",
            )
        except Exception as e:
            print(f"  [hub-push] non-fatal failure: {e}")

_tg.save_lora_adapters = save_lora_adapters_with_push
print("Hub-push hook installed.")

## 8 · Run real training (the 6-11 hour cell)

With the per-phase early-stop targets `{1: +1.5, 2: +1.2, 3: +1.0}` set above, expect:

- ~3-5 minutes per episode (6-14 env steps × Doctor.generate + 4-8 × Groq calls)
- ~1-2 minutes amortized per GRPO update (G=2 trajectories × response-token log-probs)
- **Per-group wall-clock ≈ 8-12 min** (2 episodes + 1 update)
- **Phase 1 → Phase 2 force-promote** typically lands at **episode 16-30** (sustained `+1.5` × 3 groups)
- **Phase 2 → Phase 3 force-promote** typically lands at **episode 40-70**
- **Phase 3 EARLY STOP** typically lands at **episode 70-130** (sustained `+1.0` × 3 groups)
- Reward-growth signal (rolling avg) becomes visible after ~episode 20
- If `NUM_EPISODES=120` is exhausted before Phase 3 converges, training stops at the cap and the latest checkpoint is on HF Hub — resume in a fresh session via `HF_RESUME_REPO`.

In [ ]:
metrics = train(
    num_episodes=NUM_EPISODES,
    group_size=GROUP_SIZE,
    model_name=MODEL_NAME,
    groq_api_key=os.environ.get("GROQ_NURSE_API_KEY", "") or os.environ.get("GROQ_API_KEY", ""),
    learning_rate=LEARNING_RATE,
    kl_beta=KL_BETA,
    use_wandb=USE_WANDB,
    output_dir=OUTPUT_DIR,
    dry_run=False,
    # ----- Per-phase early-stop ('train until optimal rewards are constantly received') -----
    phase_reward_targets=PHASE_REWARD_TARGETS,
    phase_min_win_rate=PHASE_MIN_WIN_RATE,
    convergence_window=CONVERGENCE_WINDOW,
    early_stop=EARLY_STOP_ENABLED,
)

## 9 · Final push: adapters + merged fp16 weights

The training loop already wrote `final_lora/` and `final_merged_fp16/` to `OUTPUT_DIR`. We push both to HF Hub so you can serve them from Vercel / a HF Space without re-running training.

In [ ]:
FINAL_LORA_DIR   = f"{OUTPUT_DIR}/final_lora"
FINAL_MERGED_DIR = f"{OUTPUT_DIR}/final_merged_fp16"

if HF_PUSH_REPO and "<your-username>" not in HF_PUSH_REPO:
    push_checkpoint_to_hub(
        FINAL_LORA_DIR, HF_PUSH_REPO,
        commit_message="final LoRA adapter",
    )
    if os.path.isdir(FINAL_MERGED_DIR):
        push_checkpoint_to_hub(
            FINAL_MERGED_DIR, f"{HF_PUSH_REPO}-merged",
            commit_message="final merged fp16",
        )
    print("Final checkpoints pushed.")
else:
    print("HF_PUSH_REPO not configured — skipping final push.")

## 10 · Per-phase training graphs (one dashboard per curriculum phase)

We render a complete 6-panel dashboard for every phase that contains episodes, plus a cross-phase overview and a phase-comparison bar chart. All PNGs are written to `er_map_grpo_checkpoints/plots/` and uploaded to HF Hub at the end of the notebook so they survive session expiry.

**Each per-phase dashboard contains:**
1. **Reward growth** — raw scatter + rolling mean (w=10) + verified rolling mean
2. **Rolling win rate** — w=20 win rate evolution within the phase
3. **Outcome distribution over time** — stacked bars (WIN/PARTIAL/INCORRECT/AMA_LOSS/FATAL_LOSS) per episode bin
4. **Reward components** — mean of each component (process / treatment / empathy / labs / etc.) within the phase
5. **GRPO update statistics** — loss + KL divergence per group update
6. **Episode length distribution** — histogram of step counts

In [ ]:
from ER_MAP.plotting import plot_per_phase_dashboards
from IPython.display import Image, display, Markdown

PLOTS_DIR = f"{OUTPUT_DIR}/plots"
written = plot_per_phase_dashboards(
    metrics_path=f"{OUTPUT_DIR}/training_metrics.json",
    output_dir=PLOTS_DIR,
)

print(f"Saved {len(written)} chart(s):")
for name, path in written.items():
    size_kb = os.path.getsize(path) / 1024
    print(f"  {name:<28s} -> {path}  ({size_kb:.0f} KB)")

# Display each chart inline in the notebook so the operator sees them
# without leaving Kaggle. Order: per-phase dashboards first (1, 2, 3),
# then the cross-phase overview, then the bar comparison.
ordered_keys = (
    sorted(k for k in written if k.startswith("phase")) +
    ["all_phases_overview", "all_phases_comparison"]
)
for key in ordered_keys:
    if key not in written:
        continue
    display(Markdown(f"### {key.replace('_', ' ').title()}"))
    display(Image(filename=written[key]))

## 10b · Push the plots to HF Hub (so they survive session expiry)

In [ ]:
if HF_PUSH_REPO and "<your-username>" not in HF_PUSH_REPO:
    push_checkpoint_to_hub(
        PLOTS_DIR, HF_PUSH_REPO,
        commit_message="per-phase training plots",
    )
else:
    print("HF_PUSH_REPO not configured — plots stay only in /kaggle/working/.")

## 11 · (Optional) Inference smoke-test on the trained model

Catches the classic "merge path looked OK but the saved model emits garbage" failure mode before the demo.

In [ ]:
from ER_MAP.training.train_grpo import generate_doctor_action, load_model_and_tokenizer
from peft import PeftModel

base_model, tok = load_model_and_tokenizer(model_name=MODEL_NAME)
trained = PeftModel.from_pretrained(base_model, FINAL_LORA_DIR)

test_obs = (
    '{"event":"episode_start","nurse_experience":"veteran",'
    '"message":"Patient with chest pain, HR 120, BP 90/60, vague history.",'
    '"soap_summary":{}}'
)
for i in range(3):
    print(f"\n--- Sample {i+1} ---")
    print(generate_doctor_action(trained, tok, test_obs, max_new_tokens=160))